## Best Hota

In [1]:
!nvidia-smi

Wed Jun 24 11:53:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# клон + установка
REPO_URL = "https://github.com/danvlak-batya/deep-sort-project.git"
PROJECT_DIR = "/content/deep-sort-project"

import os, sys
os.chdir("/content")
if os.path.exists(PROJECT_DIR):
    !rm -rf {PROJECT_DIR}
!git clone {REPO_URL} {PROJECT_DIR}
os.chdir(PROJECT_DIR)

# Проверка: в репозитории должна быть timm-версия ReID
assert os.path.exists("reid/timm_backend.py"), (
    "Старый код на GitHub! Сделайте git push с компьютера и перезапустите.")

# Ставим пакеты
pkgs = [
    "numpy>=1.22,<2.1", "opencv-python", "scipy", "filterpy",
    "PyYAML", "scikit-learn", "Pillow", "tqdm", "ultralytics", "timm",
]
for pkg in pkgs:
    print("Installing", pkg)
    !{sys.executable} -m pip install -q {pkg}

import ultralytics, timm, torch
print("\n=== Проверка OK ===")
print("torch:", torch.__version__)
print("timm:", timm.__version__)
print("ultralytics: OK")

Cloning into '/content/deep-sort-project'...
remote: Enumerating objects: 312, done.
remote: Counting objects: 100% (312/312), done.
remote: Compressing objects: 100% (167/167), done.
remote: Total 312 (delta 149), reused 287 (delta 129), pack-reused 0 (from 0)
Receiving objects: 100% (312/312), 136.30 KiB | 5.45 MiB/s, done.
Resolving deltas: 100% (149/149), done.
Installing numpy>=1.22,<2.1
/bin/bash: line 1: 2.1: No such file or directory
Installing opencv-python
Installing scipy
Installing filterpy
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
Installing PyYAML
Installing scikit-learn
Installing Pillow
Installing tqdm
Installing ultralytics
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.6 MB/s eta 0:00:00
Installing timm
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or

In [3]:
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = "/content/drive/MyDrive/Videos-CV"
import os
assert os.path.isdir(DATA_ROOT)
print(os.listdir(DATA_ROOT))

Mounted at /content/drive
['KITTI-17', 'MOT16-09', 'MOT16-11', 'PETS09-S2L1', 'TUD-Campus', 'TUD-Stadtmitte', 'results_best']


In [19]:
from utils.mot_paths import find_sequence_dir, list_image_filenames

SEQUENCE_FOLDER = "TUD-Stadtmitte" # Название папки
SEQUENCE = "TUD-Stadtmitte" # Название папки
DETECTOR = "yolov8n"
REID = "osnet_x0_25"

SEQ_DIR = find_sequence_dir(DATA_ROOT, SEQUENCE_FOLDER)
print("Кадров:", len(list_image_filenames(SEQ_DIR)))

Кадров: 179


In [20]:
import os
os.makedirs("results/demo", exist_ok=True)

!python run_tracker.py \
  --sequence_dir "{SEQ_DIR}" \
  --config configs/default.yaml \
  --detector {DETECTOR} \
  --reid {REID} \
  --output_file results/demo/{SEQUENCE}.txt

p = f"results/demo/{SEQUENCE}.txt"
print("OK:", os.path.exists(p), "size:", os.path.getsize(p) if os.path.exists(p) else 0)

Streaming output truncated to the last 5000 lines.
      177.3524205279302
    ],
    [
      71,
      2,
      558.4933221088274,
      86.38130600525218,
      79.70210946612771,
      207.96574729653207
    ],
    [
      71,
      4,
      452.02851462951367,
      95.58718141826958,
      66.967005034835,
      187.61219365048026
    ],
    [
      71,
      7,
      174.825968895082,
      95.03786859034092,
      45.98785023403891,
      157.77925997154412
    ],
    [
      72,
      1,
      385.920454421353,
      95.22467562768622,
      55.51688695161693,
      176.2237011950482
    ],
    [
      72,
      2,
      561.2332733017531,
      85.6903424081936,
      79.44018912539185,
      208.25026318617694
    ],
    [
      72,
      4,
      455.15166277119056,
      96.4555634258823,
      66.60138704089943,
      187.3172529147534
    ],
    [
      72,
      7,
      175.9270213661828,
      95.19837998418252,
      46.23629403708349,
      157.48446419224032
    ],


In [21]:
!python tools/generate_overlays.py \
  --mot_dir "{DATA_ROOT}" \
  --results_dir results/demo \
  --output_dir results/overlays \
  --sequence {SEQUENCE}

from IPython.display import Video, display
v = f"results/overlays/{SEQUENCE}_overlay.mp4"
display(Video(v, embed=True)) if os.path.exists(v) else print("Нет видео")

Saved overlay: results/overlays/TUD-Stadtmitte_overlay.mp4


In [ ]:
import os
import shutil

DATA_ROOT = "/content/drive/MyDrive/Videos-CV"
SAVE_DIR = os.path.join(DATA_ROOT, "results_best")

os.makedirs(os.path.join(SAVE_DIR, "demo"), exist_ok=True)
os.makedirs(os.path.join(SAVE_DIR, "overlays"), exist_ok=True)

for folder, name in [("results/demo", "demo"), ("results/overlays", "overlays")]:
    src = f"/content/deep-sort-project/{folder}"
    dst = os.path.join(SAVE_DIR, name)
    if os.path.isdir(src):
        for f in os.listdir(src):
            shutil.copy2(os.path.join(src, f), os.path.join(dst, f))
            print("Saved:", f)

print("\nПроверка demo:")
print(os.listdir(os.path.join(SAVE_DIR, "demo")))

Saved: PETS09-S2L1.txt
Saved: MOT16-09.txt
Saved: MOT16-11.txt
Saved: TUD-Campus.txt
Saved: KITTI-17.txt
Saved: TUD-Stadtmitte.txt
Saved: MOT16-09_overlay.mp4
Saved: MOT16-11_overlay.mp4
Saved: KITTI-17_overlay.mp4
Saved: TUD-Campus_overlay.mp4
Saved: PETS09-S2L1_overlay.mp4
Saved: TUD-Stadtmitte_overlay.mp4

Проверка demo:
['PETS09-S2L1.txt', 'MOT16-09.txt', 'MOT16-11.txt', 'TUD-Campus.txt', 'KITTI-17.txt', 'TUD-Stadtmitte.txt']


In [23]:
%cd /content/deep-sort-project

# Сбросить локальные правки от старых патчей и подтянуть GitHub
!git fetch origin
!git checkout -- eval/mot_metrics.py
!git pull origin master

# Проверка: должна быть строка DO_PREPROC = False
!grep "DO_PREPROC" eval/mot_metrics.py

# Сбросить кэш Python-модуля
import importlib
import eval.mot_metrics as mm
importlib.reload(mm)

text = open("eval/mot_metrics.py").read()
assert 'DO_PREPROC"] = False' in text, "Старая версия — git pull не сработал!"
print("mot_metrics.py — OK")

/content
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.97 KiB | 1.97 MiB/s, done.
From https://github.com/danvlak-batya/deep-sort-project
   18b65b7..cda7980  master     -> origin/master
From https://github.com/danvlak-batya/deep-sort-project
 * branch            master     -> FETCH_HEAD
Updating 18b65b7..cda7980
Fast-forward
 eval/run_baseline_from_det.py | 118 ++++++++++++++++++++++++++++++++++++++++++
 1 file changed, 118 insertions(+)
 create mode 100644 eval/run_baseline_from_det.py
        dataset_config["DO_PREPROC"] = False
mot_metrics.py — OK


In [ ]:
from eval.mot_metrics import evaluate_hota, EVAL_SEQUENCES

metrics = evaluate_hota(DATA_ROOT, RESULTS_DIR)

print("\n=== HOTA results ===")
print(f"{'Sequence':<16} {'HOTA':>8}")
print("-" * 26)
for seq in EVAL_SEQUENCES:
    if seq in metrics["per_sequence"]:
        hota = metrics["per_sequence"][seq]["HOTA"]
        print(f"{seq:<16} {hota:8.4f}")
    else:
        print(f"{seq:<16}   (missing)")
print("-" * 26)
print(f"{'Mean':<16} {metrics['mean_hota']:8.4f}")


Prepared TUD-Campus: 71 frames -> /tmp/trackeval_72ou2e3h/gt/TUD-Campus/gt/gt.txt
Prepared TUD-Stadtmitte: 179 frames -> /tmp/trackeval_72ou2e3h/gt/TUD-Stadtmitte/gt/gt.txt
Prepared KITTI-17: 145 frames -> /tmp/trackeval_72ou2e3h/gt/KITTI-17/gt/gt.txt
Prepared PETS09-S2L1: 795 frames -> /tmp/trackeval_72ou2e3h/gt/PETS09-S2L1/gt/gt.txt
Prepared MOT16-09: 525 frames -> /tmp/trackeval_72ou2e3h/gt/MOT16-09/gt/gt.txt
Prepared MOT16-11: 900 frames -> /tmp/trackeval_72ou2e3h/gt/MOT16-11/gt/gt.txt
TrackEval GT_FOLDER: /tmp/trackeval_72ou2e3h/gt
SEQ_INFO: {'TUD-Campus': 71, 'TUD-Stadtmitte': 179, 'KITTI-17': 145, 'PETS09-S2L1': 795, 'MOT16-09': 525, 'MOT16-11': 900}

Eval Config:
USE_PARALLEL         : False                         
NUM_PARALLEL_CORES   : 8                             
BREAK_ON_ERROR       : True                          
RETURN_ON_ERROR      : False                         
LOG_ON_ERROR         : /usr/local/lib/python3.12/dist-packages/error_log.txt
PRINT_RESULTS        : Fals

In [ ]:
import os
os.makedirs("results", exist_ok=True)
!python eval/run_mot.py \
  --mot_dir "{DATA_ROOT}" \
  --output_dir results/demo \
  --skip_tracking \
  --metrics_out results/metrics_hota.json
import json
with open("results/metrics_hota.json") as f:
    print(json.dumps(json.load(f), indent=2))

Prepared TUD-Campus: 71 frames -> /tmp/trackeval_66b962hs/gt/TUD-Campus/gt/gt.txt
Prepared TUD-Stadtmitte: 179 frames -> /tmp/trackeval_66b962hs/gt/TUD-Stadtmitte/gt/gt.txt
Prepared KITTI-17: 145 frames -> /tmp/trackeval_66b962hs/gt/KITTI-17/gt/gt.txt
Prepared PETS09-S2L1: 795 frames -> /tmp/trackeval_66b962hs/gt/PETS09-S2L1/gt/gt.txt
Prepared MOT16-09: 525 frames -> /tmp/trackeval_66b962hs/gt/MOT16-09/gt/gt.txt
Prepared MOT16-11: 900 frames -> /tmp/trackeval_66b962hs/gt/MOT16-11/gt/gt.txt
TrackEval GT_FOLDER: /tmp/trackeval_66b962hs/gt
SEQ_INFO: {'TUD-Campus': 71, 'TUD-Stadtmitte': 179, 'KITTI-17': 145, 'PETS09-S2L1': 795, 'MOT16-09': 525, 'MOT16-11': 900}

Eval Config:
USE_PARALLEL         : False                         
NUM_PARALLEL_CORES   : 8                             
BREAK_ON_ERROR       : True                          
RETURN_ON_ERROR      : False                         
LOG_ON_ERROR         : /usr/local/lib/python3.12/dist-packages/error_log.txt
PRINT_RESULTS        : Fals

## Baseline — оригинальный DeepSORT


In [25]:
%cd /content/deep-sort-project

!git pull origin master

# TensorFlow — для оригинального ReID mars-small128.pb
import sys
for pkg in ("tensorflow", "gdown"):
    !{sys.executable} -m pip install -q {pkg}

import os, glob, shutil
import gdown

MODEL_PATH = "resources/networks/mars-small128.pb"
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

if not os.path.exists(MODEL_PATH):
    tmp = "/tmp/deepsort_resources"
    if os.path.isdir(tmp):
        shutil.rmtree(tmp)
    print("Скачиваем mars-small128.pb из официального Drive DeepSORT...")
    gdown.download_folder(
        id="18fKzfqnqhqW3s9zwsCbnVJ5XF2JFeqMp",
        output=tmp,
        quiet=False,
    )
    hits = glob.glob(os.path.join(tmp, "**", "mars-small128.pb"), recursive=True)
    assert hits, "mars-small128.pb не найден — проверьте доступ к Google Drive"
    shutil.copy2(hits[0], MODEL_PATH)

assert os.path.getsize(MODEL_PATH) > 1_000_000, "Файл модели повреждён"
print("ReID model OK:", MODEL_PATH, "size:", os.path.getsize(MODEL_PATH))

/content/deep-sort-project
From https://github.com/danvlak-batya/deep-sort-project
 * branch            master     -> FETCH_HEAD
Already up to date.
Скачиваем mars-small128.pb из официального Drive DeepSORT...


Retrieving folder contents


Retrieving folder 1VVqtL0klSUvLnmBKS89il1EKC3IxUBVK detections
Retrieving folder 1qNWOpUtKG8GqEiL-LbBdXyvifUtcbOvc MOT16_POI_test
Processing file 1aEzvFHPK-N6hqLXMqhh3i9JJzn7WFUA3 MOT16-01.npy
Processing file 1h_ktJDBIEXaSBAA-RxKNYnL9e4fp2HPd MOT16-03.npy
Processing file 1ilOElwfYZLwQKH57HoYdXfuYhpPibfqF MOT16-06.npy
Processing file 1TajzH3GbumKmtYvKBvOtGERFGD0tStwG MOT16-07.npy
Processing file 1WB9Mi4RLVPHV4_20sVq7FdoeG5JYQ_J1 MOT16-08.npy
Processing file 1mksH9GWNT7zmcuq6rlRev8pevZz8Rfsm MOT16-12.npy
Processing file 1FVVhn_IpxQ_jkYhc0CUQHSQMm1SMTEBj MOT16-14.npy
Retrieving folder 1DcOcApOkxP3NdeIUXxVF1KNex6T6YDq3 MOT16_POI_train
Processing file 1Va__9NWU2ZCmaxIq4oIabi05NYWEOk1K MOT16-02.npy
Processing file 1EH7orgDPp7kqRY5OA0hEctcEtQnYq0Ea MOT16-04.npy
Processing file 1RCfHJx5ZoUecapbZCsgp0tCEiItvLsd8 MOT16-05.npy
Processing file 1VLOvn-mbpY0Q1rsMONQZhaEQIGEmyLQL MOT16-09.npy
Processing file 1SbMhOgYPvZ84xE8lRtXc7CLXJF86lwf4 MOT16-10.npy
Processing file 1a4w-HopWJHLFVi4e5wM_CEpv_ZgAV

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1aEzvFHPK-N6hqLXMqhh3i9JJzn7WFUA3
To: /tmp/deepsort_resources/detections/MOT16_POI_test/MOT16-01.npy
100%|██████████| 11.3M/11.3M [00:00<00:00, 159MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1h_ktJDBIEXaSBAA-RxKNYnL9e4fp2HPd
From (redirected): https://drive.google.com/uc?id=1h_ktJDBIEXaSBAA-RxKNYnL9e4fp2HPd&confirm=t&uuid=06c207b3-e280-456b-b60a-e8df64517f54
To: /tmp/deepsort_resources/detections/MOT16_POI_test/MOT16-03.npy
100%|██████████| 106M/106M [00:01<00:00, 63.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1ilOElwfYZLwQKH57HoYdXfuYhpPibfqF
To: /tmp/deepsort_resources/detections/MOT16_POI_test/MOT16-06.npy
100%|██████████| 12.0M/12.0M [00:00<00:00, 57.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1TajzH3GbumKmtYvKBvOtGERFGD0tStwG
To: /tmp/deepsort_resources/detections/MOT16_PO

ReID model OK: resources/networks/mars-small128.pb size: 11244842



Download completed


In [26]:
import os
from utils.mot_paths import find_sequence_dir, get_det_file, list_image_filenames
from eval.mot_metrics import EVAL_SEQUENCES

print("Проверка det/ и кадров на Drive:\n")
ok = True
for seq in EVAL_SEQUENCES:
    try:
        seq_dir = find_sequence_dir(DATA_ROOT, seq)
        det = get_det_file(seq_dir)
        has_det = os.path.exists(det)
        n_frames = len(list_image_filenames(seq_dir))
        status = "OK" if has_det and n_frames else "FAIL"
        print(f"  {seq:<16} det={has_det}  frames={n_frames}  [{status}]")
        ok = ok and has_det and n_frames
    except FileNotFoundError as e:
        print(f"  {seq:<16} FAIL — {e}")
        ok = False

assert ok, "Не все последовательности готовы — проверьте DATA_ROOT на Drive"
print("\nВсе 6 последовательностей готовы для baseline.")

Проверка det/ и кадров на Drive:

  TUD-Campus       det=True  frames=71  [OK]
  TUD-Stadtmitte   det=True  frames=179  [OK]
  KITTI-17         det=True  frames=145  [OK]
  PETS09-S2L1      det=True  frames=795  [OK]
  MOT16-09         det=True  frames=525  [OK]
  MOT16-11         det=True  frames=900  [OK]

Все 6 последовательностей готовы для baseline.


In [27]:
import os, shutil
from eval.mot_metrics import EVAL_SEQUENCES

BASELINE_DIR = "results/baseline"
NPY_DIR = "resources/detections"

# если baseline уже считали — можно восстановить с Drive и пропустить долгий прогон
drive_baseline = os.path.join(DATA_ROOT, "results_baseline", "demo")
os.makedirs(BASELINE_DIR, exist_ok=True)
if os.path.isdir(drive_baseline):
    for f in os.listdir(drive_baseline):
        if f.endswith(".txt"):
            shutil.copy2(
                os.path.join(drive_baseline, f),
                os.path.join(BASELINE_DIR, f),
            )
    hota_on_drive = os.path.join(DATA_ROOT, "results_baseline", "hota.json")
    if os.path.exists(hota_on_drive):
        shutil.copy2(hota_on_drive, os.path.join(BASELINE_DIR, "hota.json"))

n_tracks = len([f for f in os.listdir(BASELINE_DIR) if f.endswith(".txt")])
if n_tracks == len(EVAL_SEQUENCES):
    print(f"Baseline треки уже есть ({n_tracks}/6) — пропускаем run_baseline_from_det")
else:
    !python eval/run_baseline_from_det.py \
      --mot_root "{DATA_ROOT}" \
      --model "{MODEL_PATH}" \
      --npy_dir "{NPY_DIR}" \
      --output_dir "{BASELINE_DIR}"

Loading mars-small128 encoder...
I0000 00:00:1782305582.227959   23096 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Building TUD-Campus from det/ ...
I0000 00:00:1782305582.735642   23096 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
  frame 50 / 71
Saved: resources/detections/TUD-Campus.npy rows: 322
Baseline tracking: TUD-Campus
Processing frame 00001
Processing frame 00002
Processing frame 00003
Processing frame 00004
Processing frame 00005
Processing frame 00006
Processing frame 00007
Processing frame 00008
Processing frame 00009
Processing frame 00010
Processing frame 00011
Processing frame 00012
Processing frame 00013
Processing frame 00014
Processing frame 00015
Processing frame 00016
Processing frame 00017
Processing frame 00018
Processing frame 00019
Processing frame 00020
Processing frame 00021
Processing fra

In [28]:
import json
from eval.mot_metrics import evaluate_hota, EVAL_SEQUENCES

hota_path = os.path.join(BASELINE_DIR, "hota.json")
if not os.path.exists(hota_path):
    metrics = evaluate_hota(DATA_ROOT, BASELINE_DIR)
    with open(hota_path, "w") as f:
        json.dump(metrics, f, indent=2)
else:
    with open(hota_path) as f:
        metrics = json.load(f)

print("\n=== HOTA results (baseline) ===")
print(f"{'Sequence':<16} {'HOTA':>8}")
print("-" * 26)
for seq in EVAL_SEQUENCES:
    if seq in metrics["per_sequence"]:
        hota = metrics["per_sequence"][seq]["HOTA"]
        print(f"{seq:<16} {hota:8.4f}")
    else:
        print(f"{seq:<16}   (missing)")
print("-" * 26)
print(f"{'Mean':<16} {metrics['mean_hota']:8.4f}")


=== HOTA results (baseline) ===
Sequence             HOTA
--------------------------
TUD-Campus         0.3986
TUD-Stadtmitte     0.3675
KITTI-17           0.4341
PETS09-S2L1        0.4484
MOT16-09           0.3585
MOT16-11           0.3995
--------------------------
Mean               0.4011


In [29]:
OVERLAY_BASELINE_DIR = "results/overlays_baseline"
os.makedirs(OVERLAY_BASELINE_DIR, exist_ok=True)

!python tools/generate_overlays.py \
  --mot_dir "{DATA_ROOT}" \
  --results_dir "{BASELINE_DIR}" \
  --output_dir "{OVERLAY_BASELINE_DIR}"

print("Overlays:", sorted(os.listdir(OVERLAY_BASELINE_DIR)))

Saved overlay: results/overlays_baseline/MOT16-11_overlay.mp4
Saved overlay: results/overlays_baseline/PETS09-S2L1_overlay.mp4
Saved overlay: results/overlays_baseline/TUD-Stadtmitte_overlay.mp4
Saved overlay: results/overlays_baseline/KITTI-17_overlay.mp4
Saved overlay: results/overlays_baseline/MOT16-09_overlay.mp4
Saved overlay: results/overlays_baseline/TUD-Campus_overlay.mp4
Overlays: ['KITTI-17_overlay.mp4', 'MOT16-09_overlay.mp4', 'MOT16-11_overlay.mp4', 'PETS09-S2L1_overlay.mp4', 'TUD-Campus_overlay.mp4', 'TUD-Stadtmitte_overlay.mp4']


In [30]:
SAVE_BASELINE_DIR = os.path.join(DATA_ROOT, "results_baseline")
os.makedirs(os.path.join(SAVE_BASELINE_DIR, "demo"), exist_ok=True)
os.makedirs(os.path.join(SAVE_BASELINE_DIR, "overlays"), exist_ok=True)

for folder, name in [
    (BASELINE_DIR, "demo"),
    (OVERLAY_BASELINE_DIR, "overlays"),
]:
    src = f"/content/deep-sort-project/{folder}"
    dst = os.path.join(SAVE_BASELINE_DIR, name)
    if os.path.isdir(src):
        for f in os.listdir(src):
            shutil.copy2(os.path.join(src, f), os.path.join(dst, f))

hota_src = os.path.join(BASELINE_DIR, "hota.json")
if os.path.exists(hota_src):
    shutil.copy2(hota_src, os.path.join(SAVE_BASELINE_DIR, "hota.json"))

print("Saved to:", SAVE_BASELINE_DIR)
print(os.listdir(SAVE_BASELINE_DIR))

Saved to: /content/drive/MyDrive/Videos-CV/results_baseline
['demo', 'overlays', 'hota.json']


In [33]:
%cd /content/deep-sort-project

!python eval/run_benchmark.py \
  --mot_root "{DATA_ROOT}" \
  --output_dir results/fps_benchmark \
  --detector yolov8n \
  --reid osnet_x0_25 \
  --config configs/default.yaml

import json
with open("results/fps_benchmark/run_summary.json") as f:
    stats = json.load(f)

per_seq = {s["sequence_name"]: s["fps"] for s in stats}
mean_fps = sum(per_seq.values()) / len(per_seq)

print(f"\n{'Sequence':<16} {'FPS':>8}")
print("-" * 26)
from eval.mot_metrics import EVAL_SEQUENCES
for seq in EVAL_SEQUENCES:
    print(f"{seq:<16} {per_seq[seq]:8.2f}")
print("-" * 26)
print(f"{'Mean':<16} {mean_fps:8.2f}")

/content/deep-sort-project
Running TUD-Campus ...
ReID osnet_x0_25 -> using timm model: mobilenetv3_small_100.lamb_in1k
Frame 00050/00071
  FPS: 13.88
Running TUD-Stadtmitte ...
ReID osnet_x0_25 -> using timm model: mobilenetv3_small_100.lamb_in1k
Frame 00050/00179
Frame 00100/00179
Frame 00150/00179
  FPS: 21.93
Running KITTI-17 ...
ReID osnet_x0_25 -> using timm model: mobilenetv3_small_100.lamb_in1k
Frame 00050/00145
Frame 00100/00145
  FPS: 17.45
Running PETS09-S2L1 ...
ReID osnet_x0_25 -> using timm model: mobilenetv3_small_100.lamb_in1k
Frame 00050/00795
Frame 00100/00795
Frame 00150/00795
Frame 00200/00795
Frame 00250/00795
Frame 00300/00795
Frame 00350/00795
Frame 00400/00795
Frame 00450/00795
Frame 00500/00795
Frame 00550/00795
Frame 00600/00795
Frame 00650/00795
Frame 00700/00795
Frame 00750/00795
  FPS: 17.66
Running MOT16-09 ...
ReID osnet_x0_25 -> using timm model: mobilenetv3_small_100.lamb_in1k
Frame 00050/00525
Frame 00100/00525
Frame 00150/00525
Frame 00200/00525
Frame